# SST-5 ELMo Layer Ablation

`layer_mode`를 바꿔가며 (`0`, `1`, `2`, `"weighted"`) 동일 설정으로 학습/검증을 수행합니다.

- `0`: bottom layer
- `1`: middle layer
- `2`: top layer
- `weighted`: ELMo learnable weighted sum

In [1]:
from pathlib import Path
from datetime import datetime
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

import sst5_elmo_classifier as sst


In [ ]:
# 공통 설정
DATA_DIR = Path("SST-5")
CHECKPOINT_PATH = Path("../../checkpoints/bilm/final_model.pt").resolve()
VOCAB_FILE = Path("../../bilm/data/pretrain/elmo/vocab.txt").resolve()
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
EPOCHS = 10
BATCH_SIZE = 32
LR = 1e-3
MAX_TOKENS = 64
SEEDS = (13, 17, 23)
EARLY_STOPPING_PATIENCE = 3
LAYER_MODES = [0, 1, 2, "weighted"]
OUT_DIR = Path("ablation_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

print("device:", DEVICE)
print("checkpoint:", CHECKPOINT_PATH)
print("vocab:", VOCAB_FILE)


device: cuda
checkpoint: /home/ssai/Workspace/ELMo_repo/bilm-tf/checkpoints/bilm/tf_bilm_seed1114_e10.pt
vocab: /home/ssai/Workspace/ELMo_repo/bilm-tf/checkpoints/bilm/vocab/sst5_train_tokens.txt


In [ ]:
def train_one_layer_mode(layer_mode):
    """`sst.train_on_real_sst5`와 동일한 루프를 `layer_mode`별로 수행."""
    data_dir = sst.ensure_sst5_dataset(DATA_DIR)
    train_tsv = data_dir / "train.tsv"
    dev_tsv = data_dir / "dev.tsv"

    if not CHECKPOINT_PATH.is_file():
        raise FileNotFoundError(f"Checkpoint not found: {CHECKPOINT_PATH}")
    if VOCAB_FILE is not None and not VOCAB_FILE.is_file():
        raise FileNotFoundError(f"vocab_file not found: {VOCAB_FILE}")

    dev = torch.device(DEVICE)
    _, _, _, options = sst.load_pretrained_char_bilm_from_checkpoint(CHECKPOINT_PATH, map_location=dev)
    max_chars = int(options["char_cnn"]["max_characters_per_token"])
    n_characters = int(options["char_cnn"]["n_characters"])

    encoder = sst.CharIdEncoder(
        max_chars_per_token=max_chars,
        n_characters=n_characters,
        vocab_file=VOCAB_FILE,
    )

    train_samples = sst.read_sst5_tsv(train_tsv)
    dev_samples = sst.read_sst5_tsv(dev_tsv)
    train_ds = sst.SST5CharDataset(train_samples, max_tokens=MAX_TOKENS)
    dev_ds = sst.SST5CharDataset(dev_samples, max_tokens=MAX_TOKENS)
    collate_fn = sst.make_sst5_collate_fn(encoder)
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_fn)
    dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn)

    run_name = f"layer_{layer_mode}"
    metrics_csv = OUT_DIR / f"{run_name}_metrics.csv"
    best_by_seed_csv = OUT_DIR / f"{run_name}_best_by_seed.csv"
    ckpt_dir = OUT_DIR / "checkpoints" / run_name
    ckpt_dir.mkdir(parents=True, exist_ok=True)

    # 이전 실행 파일이 있으면 지우고 새로 기록
    for p in [metrics_csv, best_by_seed_csv]:
        if p.exists():
            p.unlink()

    print(f"\n===== layer_mode={layer_mode} =====")
    for seed in SEEDS:
        sst.set_global_seed(seed)
        bilm, num_layers, hidden_dim, _ = sst.load_pretrained_char_bilm_from_checkpoint(
            CHECKPOINT_PATH, map_location=dev
        )
        bilm.to(dev)
        model = sst.SSTClassifier(
            bilm=bilm,
            hidden_dim=hidden_dim,
            num_layers=num_layers,
            layer_mode=layer_mode,
            pooling="attention",
        ).to(dev)
        model.classifier[-1] = nn.Linear(hidden_dim, sst.NUM_LABELS).to(dev)

        optimizer = torch.optim.Adam(sst.trainable_parameters(model), lr=LR)
        criterion = nn.CrossEntropyLoss()

        best_ckpt = ckpt_dir / f"seed{seed}.pt"
        best_dev_loss = float("inf")
        best_dev_acc = 0.0
        best_epoch = 0
        patience_count = 0

        for epoch in range(1, EPOCHS + 1):
            tr_loss, tr_acc = sst.run_epoch(model, train_loader, optimizer, criterion, dev)
            with torch.no_grad():
                dv_loss, dv_acc = sst.run_epoch(model, dev_loader, None, criterion, dev)

            sst.append_epoch_metrics_csv(
                metrics_csv,
                epoch=epoch,
                train_loss=tr_loss,
                train_acc=tr_acc,
                dev_loss=dv_loss,
                dev_acc=dv_acc,
            )
            print(
                f"mode={layer_mode} seed={seed} epoch={epoch:02d} "
                f"train_loss={tr_loss:.4f} train_acc={tr_acc:.4f} "
                f"dev_loss={dv_loss:.4f} dev_acc={dv_acc:.4f}"
            )

            improved = (dv_loss < best_dev_loss) or (
                dv_loss == best_dev_loss and dv_acc > best_dev_acc
            )
            if improved:
                best_dev_loss = dv_loss
                best_dev_acc = dv_acc
                best_epoch = epoch
                patience_count = 0
                torch.save(
                    {
                        "seed": seed,
                        "epoch": epoch,
                        "layer_mode": layer_mode,
                        "model_state_dict": model.state_dict(),
                        "optimizer_state_dict": optimizer.state_dict(),
                        "best_dev_loss": best_dev_loss,
                        "best_dev_acc": best_dev_acc,
                    },
                    best_ckpt,
                )
            else:
                patience_count += 1
                if patience_count >= EARLY_STOPPING_PATIENCE:
                    break

        sst.append_best_seed_csv(
            csv_path=best_by_seed_csv,
            seed=seed,
            best_epoch=best_epoch,
            best_dev_loss=best_dev_loss,
            best_dev_acc=best_dev_acc,
            stopped_epoch=epoch,
            ckpt_path=best_ckpt,
        )

    return best_by_seed_csv


In [ ]:
summary_rows = []
for mode in LAYER_MODES:
    best_csv = train_one_layer_mode(mode)
    df = pd.read_csv(best_csv)
    summary_rows.append(
        {
            "layer_mode": str(mode),
            "mean_best_dev_acc": df["best_dev_acc"].mean(),
            "std_best_dev_acc": df["best_dev_acc"].std(ddof=0),
            "mean_best_dev_loss": df["best_dev_loss"].mean(),
            "seeds": len(df),
        }
    )

summary_df = pd.DataFrame(summary_rows).sort_values("mean_best_dev_acc", ascending=False)
display(summary_df)

ts = datetime.now().strftime("%Y%m%d_%H%M%S")
summary_path = OUT_DIR / f"layer_ablation_summary_{ts}.csv"
summary_df.to_csv(summary_path, index=False)
print("saved:", summary_path)
